# MMAction — tiếp tục train checkpoint để giảm báo nhầm khi đứng gần (v4)

Chọn **Runtime → Change runtime type → T4 GPU** (L4/A100 cũng được), sau đó **Run all** và cấp quyền Drive.
Notebook dùng dữ liệu và checkpoint đã có; không cần upload lại.

**Đây là tiếp tục học từ trọng số `interaction.pt` đã fine-tune**, không khởi tạo lại classifier ngẫu nhiên.
Backbone SlowOnly R101/AVA và detector giữ nguyên; head tương tác được cập nhật với hard negatives
đứng gần, tiếp cận, rời đi, ôm, bắt tay và đưa đồ. Head v1 được mở rộng thêm tín hiệu chuyển động/khoảng cách,
giữ nguyên logits ban đầu; checkpoint v2 được nạp strict trực tiếp.

Gói indoor có 512 video AIRTLab + SBU, chia nhóm nguồn để hai góc camera không rơi vào hai split.
Một số nhãn thời điểm trước đánh nhau có AI hỗ trợ, chưa phải human-gold. Báo cáo ghi riêng lỗi đứng gần,
recall đánh nhau, chỉ số theo sự kiện và AUC trong từng video so với motion baseline.
Checkpoint cha có thể đã thấy một phần dữ liệu public; notebook ghi audit, không coi mọi test là dữ liệu mới.

Mặc định tối đa **240 phút cho mỗi lượt train/evaluate**, 150 epoch, early stopping 20 epoch.
Không cắt bớt video để chạy cho kịp 45 phút. Cache và checkpoint lưu trực tiếp trên Drive.
Nếu bị ngắt, mở đúng notebook này, giữ `RUN_TAG` và cấu hình rồi Run all để tiếp tục.
Không dùng ngưỡng đứng gần làm veto cứng vì đánh nhau cũng có thể bắt đầu ở khoảng cách gần.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from google.colab import drive, runtime

RUN_TAG = "indoor_close_seed42"  # Đổi tên nếu muốn một thí nghiệm mới độc lập.
TRAIN_MINUTES = 240
EPOCHS = 150
LEARNING_RATE = 0.0001
AUTO_DISCONNECT_ON_SUCCESS = True
SOURCE_REVISION = "abca2e3e8f956bdbbe361be57fe9bcba60c8e96b"  # Khóa đúng source đã kiểm thử.
assert RUN_TAG and all(c.isalnum() or c in "_-" for c in RUN_TAG)
assert 1 <= TRAIN_MINUTES <= 1440
subprocess.run(["nvidia-smi"], check=True)
drive.mount("/content/drive", timeout_ms=180000)
ROOT = Path("/content/drive/MyDrive/SENTINEL")
ARTIFACTS = ROOT / "artifacts"
PACK_ZIP = ARTIFACTS / "mmaction_indoor_review_v2/indoor_review_pack.zip"
PARENT_ZIP = ARTIFACTS / "mmaction_interaction_v1/mmaction_interaction_pilot.zip"
if not PARENT_ZIP.is_file():
    PARENT_ZIP = ARTIFACTS / "mmaction_indoor_review_v2/mmaction_interaction_pilot.zip"
for path in (PACK_ZIP, PARENT_ZIP):
    assert path.is_file(), f"Không thấy đầu vào trên Drive: {path}"
SHARED = ARTIFACTS / "mmaction_warmstart_v4"
RUN = SHARED / "runs" / SOURCE_REVISION / RUN_TAG
RUN.mkdir(parents=True, exist_ok=True)
print("SOURCE:", SOURCE_REVISION, "\nRUN:", RUN, "\nPARENT:", PARENT_ZIP)

## 1. Tải source cố định và kiểm tra dữ liệu trước khi cài môi trường
Không tải dataset hay checkpoint lên GitHub. Gói gốc trên Drive giữ nguyên.

In [ ]:
def sha256(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def extract_safe(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as package:
        for item in package.infolist():
            assert (destination / item.filename).resolve().is_relative_to(destination.resolve()), item.filename
        package.extractall(destination)

workspace = Path("/content/sentinel-warmstart-source")
source_zip = Path("/content/sentinel-warmstart-source.zip")
url = f"https://codeload.github.com/TranDuon/anomaly-behavior-detector-system/zip/{SOURCE_REVISION}"
with urllib.request.urlopen(url, timeout=120) as source, source_zip.open("wb") as target:
    shutil.copyfileobj(source, target)
extract_safe(source_zip, workspace)
CODE = workspace / ("anomaly-behavior-detector-system-" + SOURCE_REVISION)
DEMO = CODE / "mmaction2_demo"
sys.path.insert(0, str(CODE))
from scripts.manifest_identity import validate_pack_manifest
MANIFEST = CODE / "benchmarks/mmaction_interaction_indoor_v2.json"
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert manifest["readiness"] == "ready"
with zipfile.ZipFile(PACK_ZIP) as archive:
    package_meta = json.loads(archive.read("PACKAGE.json"))
    manifest_identity = validate_pack_manifest(MANIFEST, archive.read("manifest.json"), package_meta)
print("Manifest JSON OK; khác định dạng:", manifest_identity["formatting_differs"])
with zipfile.ZipFile(PARENT_ZIP) as archive:
    parent_metadata = json.loads(archive.read("interaction.json"))
    assert hashlib.sha256(archive.read("interaction.pt")).hexdigest() == parent_metadata["checkpoint_sha256"]
os.chdir(CODE)
sys.path.insert(0, str(CODE))
from scripts.colab_process import run_logged
def command(args, name, timeout=2400):
    return run_logged([str(a) for a in args], RUN / (name + ".log"), timeout, revision=SOURCE_REVISION)
print("Video:", len(manifest["sources"]), "| checkpoint cha:", parent_metadata["checkpoint_sha256"])
print("Source được khóa tại:", SOURCE_REVISION)

## 2. Môi trường MMAction2 riêng (Python 3.10, Torch 2.0.1 CUDA 11.8)
Lần đầu phải cài môi trường và tải checkpoint AVA/detector chính thức; các lần sau dùng lại nếu runtime còn.

In [ ]:
os.environ.update(MPLBACKEND="Agg", CUDA_MODULE_LOADING="LAZY", OMP_NUM_THREADS="4")
ENV = Path("/content/mmaction-warmstart-env")
PY = ENV / "bin/python"
ready = ENV / ".warmstart-ready"
if not ready.is_file():
    command([sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"], "install_uv")
    command(["uv", "python", "install", "3.10"], "install_python")
    if not PY.is_file():
        command(["uv", "venv", "--python", "3.10", "--seed", ENV], "create_env")
    command([PY, "-m", "pip", "install", "-q", "numpy==1.26.4", "setuptools==69.5.1", "wheel"], "install_base")
    command([PY, "-m", "pip", "install", "-q", "torch==2.0.1+cu118", "torchvision==0.15.2+cu118",
             "--index-url", "https://download.pytorch.org/whl/cu118"], "install_torch")
    command([PY, "-m", "pip", "install", "-q", "mmcv==2.1.0", "--only-binary=mmcv",
             "-f", "https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html",
             "mmengine==0.10.7", "mmdet==3.2.0", "decord==0.6.0", "einops==0.8.1",
             "importlib-metadata", "scikit-learn==1.3.2", "opencv-python==4.8.1.78", "pytest==8.3.5"], "install_openmmlab")
if not (DEMO / "repo/.git").is_dir():
    command(["git", "clone", "--depth", "1", "--branch", "v1.2.0",
             "https://github.com/open-mmlab/mmaction2.git", DEMO / "repo"], "clone_mmaction")
command([PY, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", DEMO / "repo"], "install_mmaction")
os.environ["PYTHONPATH"] = os.pathsep.join(map(str, (CODE, DEMO, DEMO / "repo")))
command([PY, "-c", "import torch, mmcv; from mmcv.ops import roi_align; assert torch.cuda.is_available(); print(torch.__version__, mmcv.__version__, torch.cuda.get_device_name(0))"], "check_cuda")
command([PY, "-m", "pytest", DEMO / "test_warmstart_v4.py", DEMO / "test_interaction.py",
         "-q", "-p", "no:cacheprovider"], "check_training")
ready.touch()

## 3. Chuẩn bị checkpoint cha và trọng số chính thức
Kiểm SHA-256 trước khi dùng. Feature cache mới tách theo mã trích xuất/manifest, tránh dùng nhầm cache cũ.

In [ ]:
PACK = Path("/content/mmaction-warmstart-pack")
WEIGHTS = Path("/content/mmaction-warmstart-weights")
extract_safe(PACK_ZIP, PACK)
extract_safe(PARENT_ZIP, WEIGHTS)
WEIGHT_STORE = SHARED / "public_weights"
WEIGHT_STORE.mkdir(parents=True, exist_ok=True)
checks = {
    "action_slowonly_r101.pth": (
        "https://download.openmmlab.com/mmaction/detection/ava/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb/slowonly_omnisource_pretrained_r101_8x8x1_20e_ava_rgb_20201217-16378594.pth",
        "163785949c5178484012a6593d12d6fcf4d36be404fa42b5899b314e91f4703b"),
    "person_faster_rcnn_r50.pth": (
        "https://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth",
        "a5d8aa15c75b961b9c73f28a6e92af370835639a2939ff890852b052a5256a5b")}
for name, (url, expected) in checks.items():
    cached = WEIGHT_STORE / name
    if not cached.is_file() or sha256(cached) != expected:
        legacy = ARTIFACTS / "mmaction_interaction_v1/public_weights" / name
        if legacy.is_file() and sha256(legacy) == expected:
            shutil.copyfile(legacy, cached)
        else:
            temporary = cached.with_suffix(".partial")
            with urllib.request.urlopen(url, timeout=180) as source, temporary.open("wb") as target:
                shutil.copyfileobj(source, target)
            assert sha256(temporary) == expected, name + " SHA mismatch"
            temporary.replace(cached)
    shutil.copyfile(cached, WEIGHTS / name)
    print(name, "SHA OK")
print("Checkpoint cha giữ nguyên:", PARENT_ZIP)

## 4. Tiếp tục train, chọn checkpoint trên development và ngưỡng trên calibration
Tăng trọng số hard negatives; từ epoch 4 tăng lấy mẫu những negative còn bị báo nhầm.
Chọn checkpoint có xét recall/precision và FPR cảnh đứng gần; test chỉ chạy sau khi đã khóa lựa chọn.
Mục tiêu calibration: precision ≥ 0,75, recall ≥ 0,60, FPR chung và FPR đứng gần ≤ 0,10.
Không đạt thì vẫn lưu báo cáo và checkpoint để phân tích, ghi rõ gate thất bại.

In [ ]:
try:
    command([PY, DEMO / "train_warmstart_v4.py", "--manifest", MANIFEST,
             "--pack", PACK, "--weights", WEIGHTS, "--cache", SHARED / "feature_cache",
             "--output", RUN, "--minutes", TRAIN_MINUTES, "--epochs", EPOCHS,
             "--lr", LEARNING_RATE], "train", timeout=TRAIN_MINUTES * 60 + 60)
except Exception as error:
    print("Run chưa hoàn tất:", error)
    print("Giữ nguyên SOURCE_REVISION / RUN_TAG / EPOCHS / LEARNING_RATE rồi Run all để tiếp tục.")
    print("Log và checkpoint đã ghi trên Drive:", RUN)
    raise

## 5. Kết quả và file tải về
So sánh cả checkpoint cha với ngưỡng cũ và checkpoint cha được hiệu chỉnh ngưỡng trên cùng calibration.
Như vậy có thể phân biệt lợi ích học thêm với lợi ích chỉ đổi ngưỡng.
Nếu `best_epoch = 0`, chưa có epoch nào vượt checkpoint cha trên development.
`parent_exposure.json` cho biết giới hạn dữ liệu test; `coverage.json` cho biết số cửa sổ thực sự có nhãn/người.

In [ ]:
report = json.loads((RUN / "report.json").read_text(encoding="utf-8"))
assert report["status"] == "completed"
bundle = RUN / "mmaction_warmstart_v4_pilot.zip"
assert bundle.is_file()
print("Calibration đạt mục tiêu:", report["calibration"]["target_feasible"])
print("Epoch được chọn:", report["training"]["best_epoch"])
print("Audit dữ liệu cha:", json.dumps(report["parent_exposure"], ensure_ascii=False))
for name, result in report["test"].items():
    print("\n", name)
    print("Windows:", json.dumps(result["windows"], ensure_ascii=False))
    print("Đứng gần:", json.dumps(result["close_people"], ensure_ascii=False))
    print("Temporal:", json.dumps(result["temporal"], ensure_ascii=False))
print("\nCHECKPOINT:", RUN / "interaction.pt")
print("GÓI CÀI:", bundle)
print("BÁO CÁO:", RUN / "report.json")
if not report["calibration"]["target_feasible"]:
    print("Calibration chưa đạt; không cài để thay model đang dùng.")
if AUTO_DISCONNECT_ON_SUCCESS:
    print("Đã lưu kết quả lên Drive; giải phóng GPU.")
    drive.flush_and_unmount()
    runtime.unassign()